# 04 – Preparing data for the dashboard

The dashboard (`dashboard/app.py`) should load fast and run online, so it shouldn't redo all the cleaning every time.
This notebook prepares **three small, ready-to-use files** and saves them in `dashboard/data/`:

1. **`monthly_response_times.csv`** – monthly response times for England and each ambulance trust
2. **`trust_area_measures.csv`** – deprivation, age, density and population for each trust
3. **`ambulance_trusts.geojson`** – a map outline for each of the 11 trusts

For the map, ONS publishes outlines for **council areas**, not ambulance trusts. So we **merge ("dissolve")**
the council shapes into trust shapes, using the lookup table we built in notebook 02.

**New data source:** Office for National Statistics, *Counties and Unitary Authorities (December 2024) Boundaries UK BUC*
(ONS Open Geography Portal). "BUC" means *ultra-generalised*: simplified outlines that keep the file small.
Contains OS data © Crown copyright and database right.

**Run notebooks 01–03 first.**

## 1. Setup

In [ ]:
from pathlib import Path

import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import shapely

PROJECT_DIR = Path("..")
RAW_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
DASHBOARD_DATA_DIR = PROJECT_DIR / "dashboard" / "data"
DASHBOARD_DATA_DIR.mkdir(parents=True, exist_ok=True)

## 2. Monthly response times

We keep only the columns the dashboard needs: England and the 11 trusts (not the 7 NHS regions),
with call counts and total seconds so the dashboard can calculate **weighted averages** for any date range.

In [ ]:
amb = pd.read_csv(PROCESSED_DIR / "ambulance_response_times_clean.csv", parse_dates=["date"])

keep_cols = ["date", "level", "name",
             "c1_calls", "c1_total_sec", "c2_calls", "c2_total_sec"]
monthly = amb.loc[amb["level"].isin(["England", "Trust"]), keep_cols].copy()

# Drop months where a trust hadn't started reporting yet (blank rows)
monthly = monthly.dropna(subset=["c1_calls", "c2_calls"])

monthly.to_csv(DASHBOARD_DATA_DIR / "monthly_response_times.csv", index=False)
print(len(monthly), "rows,", monthly["name"].nunique(), "organisations")
print("From", monthly["date"].min().date(), "to", monthly["date"].max().date())

## 3. Area measures for each trust

From notebook 03's output we keep the area measures (not the response times, which the dashboard recalculates
for whatever date range the user picks).

In [ ]:
trust = pd.read_csv(PROCESSED_DIR / "trust_response_area_and_density.csv")

area_cols = ["trust", "population_ons", "land_km2", "imd_score", "pct_65_plus",
             "u75_mortality", "pop_weighted_density", "simple_density"]
trust_measures = trust[area_cols].rename(columns={"population_ons": "population"})
trust_measures.to_csv(DASHBOARD_DATA_DIR / "trust_area_measures.csv", index=False)
trust_measures.round(1)

## 4. Map outlines: from council areas to ambulance trusts

### Load the council boundaries
The file covers the whole UK (218 areas). We keep only the 153 English council areas in our lookup.
We match on the **area code** (`CTYUA24CD`), as always.

In [ ]:
boundary_files = sorted(RAW_DIR.glob("Counties_and_Unitary_Authorities*BUC*.geojson"))
assert boundary_files, f"No BUC boundaries GeoJSON found in {RAW_DIR.resolve()}"
councils = gpd.read_file(boundary_files[-1])
print("Areas in file:", len(councils), "| Coordinate system:", councils.crs)

lookup = pd.read_csv(PROCESSED_DIR / "council_area_to_ambulance_trust_lookup.csv")
councils = councils.merge(lookup, left_on="CTYUA24CD", right_on="area_code", how="inner")
print("English council areas matched:", len(councils))
assert len(councils) == len(lookup), "Some council areas in the lookup have no boundary"

### Dissolve into trusts
`dissolve` merges all the council shapes that share the same trust into one shape, removing the borders between them.

We do this in the **British National Grid** coordinate system (EPSG:27700), which measures in metres.
That makes merging and simplifying more accurate. Then we convert back to latitude/longitude (EPSG:4326),
which is what web maps expect.

In [ ]:
trusts_geo = (
    councils.to_crs(epsg=27700)
            .dissolve(by="trust")[["geometry"]]
            .reset_index()
)

# Simplify outlines to within 300 metres – invisible at dashboard size, but makes the file much smaller
trusts_geo["geometry"] = trusts_geo.geometry.simplify(300, preserve_topology=True)
trusts_geo = trusts_geo.to_crs(epsg=4326)

# Plotly's map drawing expects outer edges drawn clockwise. If they go the other way,
# it fills the whole world *except* the shape, and the map looks blank. orient(..., -1) makes them clockwise.
trusts_geo["geometry"] = shapely.orient_polygons(trusts_geo.geometry.values, exterior_cw=True)
trusts_geo

In [ ]:
fig, ax = plt.subplots(figsize=(7, 8))
trusts_geo.plot(ax=ax, column="trust", cmap="tab20", edgecolor="white", linewidth=0.6, legend=True,
                legend_kwds={"loc": "upper left", "bbox_to_anchor": (1, 1), "fontsize": 8})
ax.set_title("England's 11 ambulance trusts (built from council areas)")
ax.set_axis_off()
plt.tight_layout()
plt.show()

### Save the map file
We round coordinates to 4 decimal places (about 10 metres), which is plenty for a map this size and keeps the file small.

In [ ]:
out = DASHBOARD_DATA_DIR / "ambulance_trusts.geojson"
trusts_geo.to_file(out, driver="GeoJSON", COORDINATE_PRECISION=4)
print(f"Saved {out.name}: {out.stat().st_size / 1024:.0f} KB")

## 5. Summary

The dashboard now has everything it needs in `dashboard/data/`, and all three files are small enough to upload to GitHub
(so the dashboard can run online).

**Limitation:** because the trust outlines are built from whole council areas, the map borders match the lookup table
(see notebook 02), not the exact operational boundaries at a few edges.